### 

# Установка gee

In [ ]:
# !pip install geemap
# !pip install earthengine-api

# Инициализация проекта

In [1]:
import ee
ee.Authenticate()
ee.Initialize(project='isentropic-disk-489519-u7')

In [26]:
print(ee.data.getAssetRoots())

[]


# Тест апишки

In [ ]:
image = ee.Image('USGS/SRTMGL1_003')
print(image.getInfo())

In [ ]:
import folium


# Твои функции NDVI/SAVI (без изменений)
def NDVI(image):
    return image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

def SAVI(image):
    nir = image.select('SR_B5')
    red = image.select('SR_B4')
    L = 0.2
    return image.expression(
        '(1 + L) * float(nir - red)/ (nir + red + L)', 
        {'nir': nir, 'red': red, 'L': L}).rename('SAVI')

vis = {
    'min': 0, 'max': 1,
    'palette': ['FFFFFF', 'CE7E45', 'DF923D', 'F1B555', 'FCD163',
                '99B718', '74A901', '66A000', '529400', '3E8601',
                '207401', '056201', '004C00', '023B01', '012E01',
                '011D01', '011301']
}

# Коллекция
collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
              .filterDate('2023-01-01', '2023-12-31')
              .filterBounds(ee.Geometry.Point([-93.7848, 30.3252]))
              .filter(ee.Filter.lt('CLOUD_COVER', 20)))

ndvi = collection.map(NDVI).median()
savi = collection.map(SAVI).median()

# Java для CGEE
https://code.earthengine.google.com/

## NDVI

In [ ]:
// Регион (определи один раз в начале)
var roi = ee.Geometry.Rectangle([120, 70, 130, 73]);

// Загружаем композит Landsat за лето 2023
var landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
  .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
  .filterDate('2023-06-01', '2023-09-01')
  .filterBounds(roi)
  .filter(ee.Filter.lt('CLOUD_COVER', 20))
  .median()
  .clip(roi);

// NDVI = (NIR - Red) / (NIR + Red)
var ndvi = landsat.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI');

// Добавляем на карту
Map.centerObject(roi, 8);
Map.addLayer(ndvi, {min: -1, max: 1, palette: ['blue', 'white', 'green']}, 'NDVI');

## NDWI

In [ ]:
// Регион (определи один раз в начале)
var roi = ee.Geometry.Rectangle([120, 70, 130, 73]);

// Загружаем композит Landsat за лето 2023
var landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
  .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
  .filterDate('2023-06-01', '2023-09-01')
  .filterBounds(roi)
  .filter(ee.Filter.lt('CLOUD_COVER', 20))
  .median()
  .clip(roi);

// NDWI = (Green - NIR) / (Green + NIR)
var ndwi = landsat.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI');

Map.addLayer(ndwi, {min: -0.5, max: 0.5, palette: ['brown', 'white', 'blue']}, 'NDWI (вода)');

## NDWI 2

In [ ]:
// Регион (определи один раз в начале)
var roi = ee.Geometry.Rectangle([120, 70, 130, 73]);

// Загружаем композит Landsat за лето 2023
var landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
  .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
  .filterDate('2023-06-01', '2023-09-01')
  .filterBounds(roi)
  .filter(ee.Filter.lt('CLOUD_COVER', 20))
  .median()
  .clip(roi);
  
// NDWI2 = (Green - SWIR1) / (Green + SWIR1) — чувствителен к влажности листвы
var ndwi2 = landsat.normalizedDifference(['SR_B3', 'SR_B6']).rename('NDWI2');

Map.addLayer(ndwi2, {min: -0.5, max: 0.5, palette: ['yellow', 'green', 'darkgreen']}, 'NDWI (влага)');

## SAVI

In [ ]:
// Регион (определи один раз в начале)
var roi = ee.Geometry.Rectangle([120, 70, 130, 73]);

// Загружаем композит Landsat за лето 2023
var landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
  .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
  .filterDate('2023-06-01', '2023-09-01')
  .filterBounds(roi)
  .filter(ee.Filter.lt('CLOUD_COVER', 20))
  .median()
  .clip(roi);

// SAVI = ((NIR - Red) / (NIR + Red + L)) * (1 + L)
var L = 0.2; // для низкой растительности (в тундре самое то)
var nir = landsat.select('SR_B5');
var red = landsat.select('SR_B4');
var savi = landsat.expression(
  '(1 + L) * (nir - red) / (nir + red + L)',
  {nir: nir, red: red, L: L}
).rename('SAVI');

Map.addLayer(savi, {min: -1, max: 1, palette: ['blue', 'white', 'darkgreen']}, 'SAVI');

## Дельта по годам

In [ ]:
var landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
  .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
  .filterDate('2023-06-01', '2023-09-01')
  .filterBounds(roi)
  .filter(ee.Filter.lt('CLOUD_COVER', 20))
  .median()
  .clip(roi);

// Функция для получения композита за год
function getComp(year) {
  return ee.ImageCollection// Регион (определи один раз в начале)
var roi = ee.Geometry.Rectangle([120, 70, 130, 73]);

// Загружаем композит Landsat за лето 2023
('LANDSAT/LC08/C02/T1_L2')
    .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
    .filterDate(year+'-06-01', year+'-09-01')
    .filterBounds(roi)
    .filter(ee.Filter.lt('CLOUD_COVER', 20))
    .median()
    .clip(roi);
}

var comp2020 = getComp(2020);
var comp2023 = getComp(2023);

// Считаем NDVI для каждого
var ndvi2020 = comp2020.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI_2020');
var ndvi2023 = comp2023.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI_2023');

// Разность (позеленение или постарение)
var ndviDiff = ndvi2023.subtract(ndvi2020).rename('NDVI_diff');

Map.addLayer(ndvi2020, {min: -1, max: 1, palette: ['blue', 'white', 'green']}, 'NDVI 2020');
Map.addLayer(ndvi2023, {min: -1, max: 1, palette: ['blue', 'white', 'green']}, 'NDVI 2023');
Map.addLayer(ndviDiff, {min: -0.2, max: 0.2, palette: ['red', 'white', 'green']}, 'Изменение NDVI');

## Выгрузка с апишки

In [ ]:
region_coords = [125, 69, 127, 71]  # Якутия, севернее Якутска, зона сплошной мерзлоты
region = ee.Geometry.Rectangle(
    coords=region_coords,
    proj="EPSG:4326",
    geodesic=False
)

# region = ee.Geometry.Rectangle(coords=region_coords, proj="EPSG:4326")

# Создаём сетку (ячейки 500×500 м в проекции EPSG:3857)
# 1.2. Сетка 500x500 м в метрической проекции
# В GEE удобно использовать coveringGrid
proj = ee.Projection('EPSG:3857')        # Web Mercator, масштаб в метрах
cell_size_m = 500                        # 500×500 м ячейки
# можно вместо 500 поставить 1000 для 1×1 км
grid_fc = region.coveringGrid(proj, scale=cell_size_m)
# grid_fc — FeatureCollection из прямоугольников 500×500 м, покрывающих регион
print("Количество ячеек сетки:", grid_fc.size().getInfo())

print("Регион задан:", region.getInfo()["coordinates"])

функции

In [ ]:
# --- 2. ШАБЛОН ДЛЯ ИНДЕКСОВ (NDVI и другие) ---

def add_ndvi_landsat8(image):
    """Добавляет NDVI к Landsat 8 C2/T1_L2 (SR_B5 = NIR, SR_B4 = Red)."""
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    return image.addBands(ndvi)

def add_other_indices_landsat8(image):
    """Шаблон: добавляет другие индексы (можно расширять)."""
    # NDVI
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    # NDWI (вода) – NIR и SWIR
    ndwi = image.normalizedDifference(['SR_B5', 'SR_6']).rename('NDWI')
    # EVI (улучшенный индекс зелёности) – пример формулы
    # EVI = 2.5 * (NIR - Red) / (NIR + 6*Red - 7.5*Blue + 1)
    # здесь условно, можно подставить реальные коэффициенты
    #evi = image.expression(
    #    '2.5 * (NIR - Red) / (NIR + 6*Red - 7.5*Blue + 1)',
    #    {
    #        'NIR': image.select('SR_B5'),
    #        'Red': image.select('SR_B4'),
    #        'Blue': image.select('SR_2')
    #    }
    #).rename('EVI')
    return image.addBands(ndvi).addBands(ndwi)

# Пример Landsat‑коллекции (Landsat 8 Collection 2 Tier 1 Level 2)
def get_landsat_ndvi_collection(year, start_month=6, end_month=8):
    """Возвращает коллекцию Landsat NDVI за заданный год (лето, вегетационный сезон)."""
    start = ee.Date(f'{year}-{start_month:02d}-01')
    end   = ee.Date(f'{year}-{end_month:02d}-31').advance(1, 'day')

    coll = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
            .filterDate(start, end)
            .filterBounds(region)
            .map(add_ndvi_landsat8)  # или add_other_indices_landsat8
            .select('NDVI'))
    return coll

In [ ]:
year = 2022
dataset = get_landsat_ndvi_collection(year, 6, 8)

print("Количество изображений в коллекции:", dataset.size().getInfo())  # Должно быть > 0

if dataset.size().getInfo() == 0:
    print("Ошибка: нет снимков в 2020‑06–08 в этом регионе для выбранного Landsat‑продукта.")

In [ ]:
# --- 3. СРЕДНИЙ NDVI ЗА ГОД ---
year = 2022

dataset = get_landsat_ndvi_collection(year, start_month=6, end_month=8)
ndvi_mean_img = dataset.mean()  # карта среднего NDVI за лето

# Считаем среднее NDVI по региону (одно число)
ndvi_stats = ndvi_mean_img.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=region,
    scale=30,          # 30 м — разрешение Landsat
    maxPixels=1e9,
    crs='EPSG:4326'
)
ndvi_value = ndvi_stats.get('NDVI')

# --- 4. ФОРМИРОВАНИЕ FEATURE (GeoJSON‑объект) ---

# Геометрия как GeoJSON (наш полигон)
region_geojson = region.getInfo()
region_geojson['type'] = 'Polygon'

# Создаём Feature с NDVI и шаблоном под другие индексы
feature = ee.Feature(
    region,
    properties={
        'year': year,
        'ndvi': ndvi_value,
        # Шаблон для других индексов (можно потом заполнять)
        'ndwi': ee.Number(0),
        'evi':  ee.Number(0),
        # Шаблон для отношений по годам (пока пусто)
        'ndvi_ratio_2015_2020': ee.Number(0),
        'ndvi_ratio_2020_2025': ee.Number(0),
    }
)

fc = ee.FeatureCollection([feature])

print("Пример свойств:", fc.first().getInfo())

In [ ]:
# --- 5. ЭКСПОРТ В GOOGLE DRIVE (GeoJSON) ---

# Имя задачи и файла
task = ee.batch.Export.table.toDrive(
    collection=fc,
    description='Landsat_Tundra_NDVI_Yakutia',
    fileNamePrefix='L8_ndvi_yak_2020',
    fileFormat='GeoJSON',
    selectors=['year', 'ndvi', 'ndwi', 'evi',
               'ndvi_ratio_2015_2020', 'ndvi_ratio_2020_2025']
)

task.start()

print("Задача запущена, проверьте Google Drive -> 'Earth Engine Exports'")
print("Для Landsat NDVI область:", region_coords)

In [ ]:
task.status()

## Выгрузка по сетке

In [ ]:
region_coords = [125, 69, 127, 71]  # Якутия, севернее Якутска, зона сплошной мерзлоты
region = ee.Geometry.Rectangle(
    coords=region_coords,
    proj="EPSG:4326",
    geodesic=False
)

# region = ee.Geometry.Rectangle(coords=region_coords, proj="EPSG:4326")

# Создаём сетку (ячейки 500×500 м в проекции EPSG:3857)
# 1.2. Сетка 500x500 м в метрической проекции
# В GEE удобно использовать coveringGrid
proj = ee.Projection('EPSG:3857')        # Web Mercator, масштаб в метрах
cell_size_m = 1000                        # 500×500 м ячейки
# можно вместо 500 поставить 1000 для 1×1 км
grid_fc = region.coveringGrid(proj, scale=cell_size_m)
# grid_fc — FeatureCollection из прямоугольников 500×500 м, покрывающих регион
print("Количество ячеек сетки:", grid_fc.size().getInfo())

print("Регион задан:", region.getInfo()["coordinates"])

In [13]:
a = [147.48, 70.83]
b = [a[0]-1, a[1]-1, a[0]+1, a[1]+1]
b

[146.48, 69.83, 148.48, 71.83]

In [ ]:
region_lists = [[125, 69, 127, 71]
, [154.7617276, 61.506373, 156.7617276, 63.506373] #'Омсукчан' '[155.7617276, 62.506373]' есть 
, [149.0977, 58.685, 151.0977, 60.685]  #'Армань' '[150.0977, 59.685]' есть
, [148.626, 60.117, 150.626, 62.117] #'Усть_омчуг' '[149.626, 61.117]'
, [147.17, 61.778, 149.17, 63.778] #'Сусуман' '[148.170, 62.778]'
, [144.602, 62.838, 146.602, 64.838] #'Делянкир' '[145.602, 63.838]'
, [142.184, 62.251, 144.184, 64.251] #'Оймякон' '[143.184, 63.251]'
, [140.893, 63.047, 142.893, 65.047] #'Юрты' '[141.893, 64.047]'
, [138.63, 62.241, 140.63, 64.241] #'Восточная' '[139.63, 63.241]'
, [135.87, 61.79, 137.87, 63.79] #'Теплый_ключ' '[136.87, 62.79]'
, [132.558, 61.403, 134.558, 63.403] #'Ытык_Кюель' '[133.558, 62.403]'
, [131.423, 60.97, 133.423, 62.97] # 'Чурапча' '[132.423, 61.97]'
, [131.0299, 59.9033, 133.0299, 61.9033] # 'Амга' '[132.0299, 60.9033]'
, [133.443, 59.37, 135.443, 61.37] # 'Усть_мая' '[134.443, 60.37]'
, [136.679, 58.76, 138.679, 60.76] # 'Югоренок' '[137.679, 59.76]'
, [128.66, 61.015, 130.66, 63.015] # 'Якутск' '[129.66, 62.015]'
, [128.166, 60.49, 130.166, 62.49] # 'Покровск' '[129.166, 61.49]'
, [125.7315, 61.077, 127.7315, 63.077] # 'Бердигестях' '[126.7315, 62.077]'
, [126.4223, 62.9629, 128.4223, 64.96289999999999] # 'Сангары' '[127.4223, 63.9629]'
, [120.4726, 62.7035, 122.4726, 64.70349999999999] # 'Вилюйск' '[121.4726, 63.7035]'
, [117.3378, 62.3, 119.3378, 64.3] # 'Нюрба' '[118.3378, 63.30]'
, [116.59, 61.12, 118.59, 63.12] # 'Сунтар' '[117.59, 62.12]'
, [115.1718, 61.25, 117.1718, 63.25] # 'Крестях' '[116.1718, 62.25]'
, [112.9599, 61.5454, 114.9599, 63.5454] # 'Мирный' '[113.9599, 62.5454]'
, [113.3971, 60.42, 115.3971, 62.42] # 'Дорожный' '[114.3971, 61.42]'
, [111.509, 63.599999999999994, 113.509, 65.6] # 'Хабардино' '[112.509, 64.6]'
, [112.39, 61.96, 114.39, 63.96] # 'Вилюйская_плотина_svetly_1' '[113.39, 62.96]'
, [111.504, 62.01, 113.504, 64.00999999999999] # 'Вилюйская_плотина_cheer1' '[112.504, 63.01]'
, [111.49, 63.44, 113.49, 65.44] # 'Вилюйская_плотина_Anyiaah_2' '[112.49, 64.44]'
, [111.32, 65.38, 113.32, 67.38] # 'Удачный_1' '[112.32, 66.38]'
, [99.32, 63.16, 101.32, 65.16] # 'Красноярск_tura '[100.32, 64.16]'
, [86.95, 64.78, 88.95, 66.78] # 'Красноярск_turukhansk '[87.95, 65.78]'

, [117.2, 55.76, 119.2, 57.76] # 'Belenskiy-4' [118.2, 56.76]
, [75.5, 65.18, 77.5, 67.18] # 'Urengoy05-03' [76.5, 66.18]
, [71.862, 64.31, 73.862, 66.31] # 'Nadym_THA' [72.862, 65.31]
, [71.87, 64.67, 73.87, 66.67] # 'Nadym8_10' [72.87, 65.67]
, [63.89, 66.37, 65.89, 68.37] # 'Vorkuta_SH-15' [64.89, 67.37]
, [61.33, 66.33, 63.33, 68.33] # 'VorkutaRU0040' [62.33, 67.33]
, [60.5, 66.95, 62.5, 68.95] # 'Vorkuta_37' [61.50, 67.95]
, [54.14, 66.57, 56.14, 68.57] # 'Shapkina_08' [55.14, 67.57]
, [176.39, 63.599999999999994, 178.39, 65.6] # 'Dionisiy-111' [177.39, 64.6]
, [170.94, 66.48, 172.94, 68.48] # 'Lake_elgygytgyn' [171.94, 67.48]
, [158.07, 67.63, 160.07, 69.63] # 'Duvanny_Yar_2_08' [159.07, 68.63]
, [146.48, 69.83, 148.48, 71.83] # 'Kytalik_GI_01_7m' [147.48, 70.83]


]

In [37]:
task.status()

{'state': 'COMPLETED',
 'description': '147;70',
 'priority': 100,
 'creation_timestamp_ms': 1777490505529,
 'update_timestamp_ms': 1777490708985,
 'start_timestamp_ms': 1777490509732,
 'task_type': 'EXPORT_FEATURES',
 'destination_uris': ['https://drive.google.com/#folders/1_jUdMkFxcfRuGz6PWPh1kQGn3GbupBHn'],
 'attempt': 1,
 'batch_eecu_usage_seconds': 8.157474517822266,
 'id': 'GEA54P76NZS24CZB2ZMY3Y2H',
 'name': 'projects/isentropic-disk-489519-u7/operations/GEA54P76NZS24CZB2ZMY3Y2H'}

In [18]:
tasks = ee.data.listOperations()
running = [t for t in tasks if t.get('metadata', {}).get('state') == 'RUNNING']
completed = [t for t in tasks if t.get('metadata', {}).get('state') == 'COMPLETED']


In [19]:
completed

[]

In [38]:
# 2. Регион: Якутия, высокомерзлотная зона
name = 'Belenskiy-4_2022'
discr = '118;56'

region_coords = [117.2, 55.76, 119.2, 57.76] # '[113.39, 62.96]'
  # [запад, юг, восток, север]
region = ee.Geometry.Rectangle(
    coords=region_coords,
    proj="EPSG:4326"
)

print("Регион задан:", region.getInfo())

# 3. Сетка 1000×1000 м в Web Mercator (EPSG:3857)
proj = ee.Projection('EPSG:3857')
cell_size_m = 1000  # 1000×1000 м ячейки

grid_fc = region.coveringGrid(proj, scale=cell_size_m)
print("Количество ячеек сетки:", grid_fc.size().getInfo())

# 4. Год и диапазон дат
year = 2022
start = ee.Date(f'{year}-01-01')
end   = ee.Date(f'{year}-12-31').advance(1, 'day')


# 5. Функция добавления нескольких индексов (NDVI, NDWI)
def add_other_indices_landsat8(image):
    # NDVI: (NIR - Red) / (NIR + Red)
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

    # NDWI: (NIR - SWIR) / (NIR + SWIR) — вода/лужи, термокарст
    ndwi_1 = image.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI_1')
    ndwi_2 = image.normalizedDifference(['SR_B3', 'SR_B6']).rename('NDWI_2')

    # SAVI (L = 0.2, NIR = B5, Red = B4)
    nir = image.select('SR_B5')
    red = image.select('SR_B4')
    L = 0.2
    savi = image.expression(
        '(1 + L) * (nir - red) / (nir + red + L)',
        {'nir': nir, 'red': red, 'L': L}
    ).rename('SAVI')
    
    return image.addBands([ndvi, ndwi_1, ndwi_2, savi])

# 6. Сборка Landsat‑8 Collection 2 Tier 1 Level 2
dataset = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
           .filterDate(start, end)
           .filterBounds(region)
           .map(add_other_indices_landsat8)
           .select(['NDVI', 'NDWI_1', 'NDWI_2', 'SAVI']))


# # 2. Средняя карта по NDVI/NDWI/SAVI
mean_img = dataset.mean()  # 3–4 бэнда
# # 2.1. Средняя LST за лето (один бэнд)
# lst = (
#     dataset.select('ST_B10') 
#     .mean()                 # 1 бэнд
#     .rename('LST')          # 1 имя → 1 бэнд — нормально
# )
# # 2.2. T и ALT (тоже однобэндовые)
# T = (
#     lst.clamp(0, 60)
#     .multiply(121)          # days
#     .rename('T')
# )
# C = 0.03
# alt = T.pow(0.5).multiply(C).rename('ALT')
# # 3. Финальный слой: базовые индексы + LST + T + ALT
# mean_img = (
#     mean_img
#     .addBands([lst, T, alt])
# )
# print("Бэнды в коллекции:", dataset.first().bandNames().getInfo())

# 7. Проверяем, есть ли снимки
n_images = dataset.size().getInfo()
if n_images == 0:
    raise Exception(f"Ошибка: нет снимков в {year} для этого региона")

print("Количество снимков в 2022:", n_images)
print("Финальные бэнды:", mean_img.bandNames().getInfo())

# 8. Средняя карта по NDVI и NDWI за год
# mean_img = dataset.mean().addBands([lst, T, alt])
# print("Бэнды средней карты:", mean_img.bandNames().getInfo())

# 9. Считаем средние значения в каждой ячейке 1000×1000 м
ndvi_ndwi_reduced = mean_img.reduceRegions(
    collection=grid_fc,
    reducer=ee.Reducer.mean(),
    scale=30,        # 30 м — разрешение Landsat
    crs='EPSG:3857'
)

# print("Пример свойств первой ячейки (getInfo):",
#       ndvi_ndwi_reduced.first().getInfo()['properties'])


# 10. Переименуем NDVI/NDWI -> ndvi/ndwi и добавим год + шаблоны
def attach_year_properties(feature):
    # 10.1. Извлекаем NDVI и NDWI (через safe‑чтение)
    ndvi = feature.get('NDVI')  # NDVI или mean_NDVI
    ndwi_1 = feature.get('NDWI_1')  # NDWI или mean_NDWI
    ndwi_2 = feature.get('NDWI_2')  # NDWI или mean_NDWI
    savi = feature.get('SAVI')  # SAVI или mean_SAVI
    # lst = feature.get('LST')  # SAVI или mean_SAVI
    # t = feature.get('T')  # SAVI или mean_SAVI
    # alt = feature.get('ALT')  # SAVI или mean_SAVI

    # 10.2. Устанавливаем короткие имена и убираем старые
    feature = feature \
        .set('ndvi', ndvi).set('NDVI', None) \
        .set('ndwi_1', ndwi_1).set('NDWI_1', None) \
        .set('ndwi_2', ndwi_2).set('NDWI_2', None) \
        .set('savi', savi).set('SAVI', None) \
        # .set('lst', lst).set('LST', None) \
        # .set('t', t).set('T', None) \
        # .set('alt', alt).set('ALT', None)

    # 10.3. Добавляем год и шаблон для других индексов и отношений
    return feature.set({
        'year': year,
        'ndvi': ndvi,
        'ndwi_1': ndwi_1,
        'ndwi_2': ndwi_2,
        'savi':  savi,
        # 'lst': lst,
        # 't': t,
        # 'alt': alt,
    })

fc_with_ndvi_ndwi = ndvi_ndwi_reduced.map(attach_year_properties)
print("Финальные свойства первой ячейки (getInfo):",
      fc_with_ndvi_ndwi.first().getInfo()['properties'])


# 11. Экспорт в Google Drive как GeoJSON с:
#     - ndvi, ndwi, год, шаблонами,
#     - и геометрией (через .geo)
task = ee.batch.Export.table.toDrive(
    collection=fc_with_ndvi_ndwi,
    description=discr,
    fileNamePrefix=name,
    folder='GEE_exports',        # folder вместо driveFolder
    fileFormat='GeoJSON',
    selectors=[
        'year',
        'ndvi',
        'ndwi_1',
        'ndwi_2',
        'savi',
        # 'lst',
        # 't',
        # 'alt',
        '.geo'    # ← ключевое поле для сохранения geometry в GeoJSON
    ]
)

task.start()
print("Задача запущена.")
print("Проверьте Google Drive -> папка 'GEE_exports' -> файл '{name}'")
print("Регион (запад, юг, восток, север):", region_coords)
print(name)

Регион задан: {'type': 'Polygon', 'coordinates': [[[117.2, 55.76], [119.2, 55.76], [119.2, 57.76], [117.2, 57.76], [117.2, 55.76]]]}
Количество ячеек сетки: 91204
Количество снимков в 2022: 142
Финальные бэнды: ['NDVI', 'NDWI_1', 'NDWI_2', 'SAVI']
Финальные свойства первой ячейки (getInfo): {'ndvi': 0.11903588775044485, 'ndwi_1': -0.10567540489986514, 'ndwi_2': 0.043546940077014415, 'savi': 0.1428416746938224, 'year': 2022}
Задача запущена.
Проверьте Google Drive -> папка 'GEE_exports' -> файл '{name}'
Регион (запад, юг, восток, север): [117.2, 55.76, 119.2, 57.76]
Belenskiy-4_2022


In [ ]:
# Глянем ка что получилось
import geopandas as gpd
import pandas as pd

# 1. Загрузка GeoJSON
file_path = 'ndvi_ndwi_1000m_yak_2022.geojson'
gdf = gpd.read_file(file_path)

print("Количество ячеек:", len(gdf))
print("\nСтолбцы свойств:")
print(gdf.columns.tolist())

print("\nПример записи:")
print(gdf.head(30))
# print(gdf.head(3)[['year', 'ndvi', 'ndwi', 'geometry']])

In [ ]:
# 1. Статусы геометрий
print("is_empty для каждой геометрии (10 строк):")
print(gdf['geometry'].is_empty.head(10))

print("\n`is_empty` YES/NO:")
print(gdf['geometry'].is_empty.value_counts())

print("\n`is_valid` YES/NO:")
print(gdf['geometry'].is_valid.value_counts())

# 2. Отметим, какие строки имеют `is_empty`
empty_mask = gdf['geometry'].is_empty
print("\nКоличество строк с empty geometry:", empty_mask.sum())

# 3. Отметим, какие строки имеют `is_valid = False`
invalid_mask = ~gdf['geometry'].is_valid
print("Количество строк с invalid geometry:", invalid_mask.sum())

In [ ]:
# Проверка x и y отдельно
print("centroids_4326.x.isna().sum():", centroids_4326.x.isna().sum())
print("centroids_4326.y.isna().sum():", centroids_4326.y.isna().sum())

print("centroids_4326.x.head():", centroids_4326.x.head())
print("centroids_4326.y.head():", centroids_4326.y.head())

# Найдём строки с Point(NaN, NaN)
nan_centroids_mask = centroids_4326.x.isna() & centroids_4326.y.isna()
print("Строк с Point(NaN, NaN):", nan_centroids_mask.sum())

In [ ]:
# 1. Проверяем, что геометрии валидные и не пустые
print("gdf_4326.is_valid.sum():", gdf_4326.is_valid.sum())
print("len(gdf_4326):", len(gdf_4326))

# 2. Проверим, что координаты геометрий не `NaN`
def check_nan_coords(geom):
    try:
        coords = np.array(geom.exterior.coords)
        return np.isnan(coords).any()
    except:
        return True

gdf_4326['has_nan_coords'] = gdf_4326.geometry.apply(check_nan_coords)
print("Строк с NaN‑координатами в геометриях:", gdf_4326['has_nan_coords'].sum())

In [ ]:
import geopandas as gpd

# 1. Загрузка без доверия к CRS
gdf = gpd.read_file('Армань.geojson')

print("gdf.crs ДО:", gdf.crs)
print("gdf.head():")
print(gdf[['year', 'ndvi', 'geometry']].head())

# 2. ЯВНО задаём: координаты в Web‑Mercator
gdf = gdf.set_crs('EPSG:3857', allow_override=True)

print("gdf.crs ПОСЛЕ set_crs:", gdf.crs)

# 3. Настоящая конвертация в градусы
gdf_4326 = gdf.to_crs('EPSG:4326')

print("gdf_4326.crs:", gdf_4326.crs)
print("gdf_4326.head():")
print(gdf_4326[['year', 'ndvi', 'geometry']].head())

# 4. Сохранение для Kepler
gdf_4326.to_file('Армань_tune.geojson', driver='GeoJSON')

In [ ]:
# Проверка крайних координат
print("Типичные координаты (x/lon):", gdf_4326.geometry.bounds['minx'].head())
print("Типичные координаты (y/lat):", gdf_4326.geometry.bounds['miny'].head())

# Если видишь что‑то вроде 125–127 (long), 69–71 (lat) — это норма.

## Шаблон для отношений NDVI по годам

Если позже подключишь Sentinel‑2:

замени коллекцию на COPERNICUS/S2_SR,

NIR = B8, Red = B4, SWIR = B11 и т.п.,

остальная логика (средний NDVI по региону → Feature → GeoJSON) одна и та же.

In [ ]:
import geopandas as gpd
import folium
import branca.colormap as cm

# 1.1. Загружаем GeoJSON с ячейками 500×500 м
file = 'ndvi_grid_500m_yak_2022.geojson'
gdf = gpd.read_file(file)

# 1.2. Проверим, что NDVI есть и тип геометрии
print("Столбцы:", gdf.columns.tolist())
print("Первые строки:")
print(gdf.head(3)[['year', 'ndvi', 'geometry']])

# 1.3. Проверим тип геометрии
print("Тип геометрии:", gdf.geometry.geom_type.unique())

In [ ]:
# 2.1. Определим центр карты (по центрам полигонов)
centroids = gdf.geometry.centroid
center_lat = centroids.y.mean()
center_lon = centroids.x.mean()

# 2.2. Создаём карту
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles='OpenStreetMap'
)

# 2.3. Цветовая шкала для NDVI
min_ndvi = gdf['ndvi'].min()
max_ndvi = gdf['ndvi'].max()
colormap = cm.LinearColormap(
    colors=['lightcoral', 'yellow', 'limegreen'],
    vmin=min_ndvi, vmax=max_ndvi,
    caption='NDVI 2022'
)

# 2.4. Добавляем ячейки на карту
for idx, row in gdf.iterrows():
    # Берём геометрию ячейки
    geom = row.geometry
    ndvi_val = row['ndvi']

    # 2.4.1. Визуализация ячейки как Polygon (polyline)
    folium.Polygon(
        locations=[list(geom.exterior.coords)],
        color=colormap(ndvi_val),
        fill=True,
        fill_color=colormap(ndvi_val),
        fill_opacity=0.6,
        weight=1,
        popup=f"NDVI: {ndvi_val:.3f}"
    ).add_to(m)

# 2.5. Добавляем цветовую шкалу
colormap.add_to(m)

# 2.6. Сохраняем
m.save('ndvi_500m_yak_2022_map.html')
print("Карта сохранена как ndvi_500m_yak_2022_map.html")

## Отншоение одного к другому

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Загрузка геоданных за 2022 и 2025
gdf_2022 = gpd.read_file('Вилюйская_плотина_svetly_1_2022.geojson')
gdf_2025 = gpd.read_file('Вилюйская_плотина_svetly_1_2025.geojson')

print("2022:", len(gdf_2022))
print("2025:", len(gdf_2025))

# 2. Убедимся, что CRS одинаковый
gdf_2022 = gdf_2022.set_crs('EPSG:3857', allow_override=True)
gdf_2025 = gdf_2025.set_crs('EPSG:3857', allow_override=True)

gdf_2022 = gdf_2022.to_crs('EPSG:4326')
gdf_2025 = gdf_2025.to_crs('EPSG:4326')

# 3. Соединим ячейки по геометрии (пересечение/перекрытие)
merged = gpd.sjoin(
    gdf_2022[['geometry', 'ndvi', 'ndwi_2', 'savi']],
    gdf_2025[['geometry', 'ndvi', 'ndwi_2', 'savi']],
    how='inner',
    predicate='intersects'
)

print("Совпадающих ячеек:", len(merged))

# 4. Переименуем колонки
merged = merged.rename(
    columns={
        'ndvi_left': 'ndvi_2022',
        'ndvi_right': 'ndvi_2025',
        'ndwi_2_left': 'ndwi_2022',
        'ndwi_2_right': 'ndwi_2025',
        'savi_left': 'savi_2022',
        'savi_right': 'savi_2025',
    }
)

# 5. Вычисляем изменения
merged['ndvi_change'] = merged['ndvi_2025'] - merged['ndvi_2022']
merged['ndwi_change'] = merged['ndwi_2025'] - merged['ndwi_2022']
merged['savi_change'] = merged['savi_2025'] - merged['savi_2022']

# 6. Эмпирический флаг "мерзлота тает" (по твоей логике)
# 6.1. рост влажности (NDWI)
merged['is_thawing_water'] = merged['ndwi_change'] > 0.05

# 6.2. падение растительности (NDVI/SAVI)
merged['is_thawing_veg_ndvi'] = merged['ndvi_change'] < -0.05
merged['is_thawing_veg_savi'] = merged['savi_change'] < -0.05

# 6.3. Суммарный флаг таяния
merged['is_thawing'] = (
    merged['is_thawing_water'] & 
    (merged['is_thawing_veg_ndvi'] | merged['is_thawing_veg_savi'])
)

# 7. Сохраняем GeoJSON для Kepler.gl
gdf_thaw = merged.copy()
gdf_thaw = gdf_thaw[gdf_thaw.columns.dropna()]  # убираем None, если есть
gdf_thaw['label'] = gdf_thaw['is_thawing'].astype(int)  # 0 = нет, 1 = тает

gdf_thaw.to_file('thaw_labels_1000m_yak_2022_2025.geojson', driver='GeoJSON')
print("GeoJSON для Kepler.gl сохранён как thaw_labels_1000m_yak_2022_2025.geojson")

2022: 109491
2025: 109491
Совпадающих ячеек: 981135
GeoJSON для Kepler.gl сохранён как thaw_labels_1000m_yak_2022_2025.geojson
